# Segmentation and Recommendation Model

**Objective:** Segment customers by behavior and create a transparent, reproducible rule-based churn-risk score.

**Important:** The score is a prioritization index, not a probability prediction and not evidence of causation.


## 1. Load the customer churn data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("customer_churn_sample.csv")
print("Shape:", df.shape)
display(df.head())


## 2. Choose features

The model uses:
- `ContractType` — account/contract feature
- `SupportTickets` — recent behavioral/support signal
- `TenureMonths` — lifecycle feature
- `MonthlyCharges` — account-value feature

These are interpretable and available in the supplied dataset.


## 3. Create customer segments

Business-rule segments:
1. **High-touch monthly:** Month-to-Month + 3+ support tickets
2. **New high-support:** ≤6 months tenure + 2+ support tickets
3. **Stable contracted:** One/Two Year + 0–1 support tickets
4. **High-value support:** charge ≥ sample median + 2+ support tickets
5. **Standard:** remaining customers


In [ ]:
def tenure_band(x):
    if x <= 6: return "0–6 months"
    if x <= 12: return "7–12 months"
    if x <= 24: return "13–24 months"
    return "25+ months"

df["TenureBand"] = df["TenureMonths"].apply(tenure_band)

charge_threshold = df["MonthlyCharges"].median()

def segment(row):
    if row["ContractType"] == "Month-to-Month" and row["SupportTickets"] >= 3:
        return "High-touch monthly"
    if row["TenureMonths"] <= 6 and row["SupportTickets"] >= 2:
        return "New high-support"
    if row["ContractType"] in ["One Year","Two Year"] and row["SupportTickets"] <= 1:
        return "Stable contracted"
    if row["MonthlyCharges"] >= charge_threshold and row["SupportTickets"] >= 2:
        return "High-value support"
    return "Standard"

df["Segment"] = df.apply(segment, axis=1)
display(df[["CustomerID","Segment"]])


## 4. Transparent churn-risk score

| Rule | Points |
|---|---:|
| Month-to-Month contract | +2 |
| 3+ support tickets | +2 |
| Tenure ≤6 months | +1 |
| Monthly charge ≥ sample median | +1 |

Maximum possible score = **6**.

Risk bands:
- High = 4–6
- Medium = 2–3
- Low = 0–1


In [ ]:
df["Score_MonthlyContract"] = (df["ContractType"]=="Month-to-Month").astype(int)*2
df["Score_HighSupport"] = (df["SupportTickets"]>=3).astype(int)*2
df["Score_ShortTenure"] = (df["TenureMonths"]<=6).astype(int)
df["Score_HighCharge"] = (df["MonthlyCharges"]>=charge_threshold).astype(int)

df["ChurnRiskScore"] = (
    df["Score_MonthlyContract"] +
    df["Score_HighSupport"] +
    df["Score_ShortTenure"] +
    df["Score_HighCharge"]
)

def risk_band(score):
    if score >= 4: return "High"
    if score >= 2: return "Medium"
    return "Low"

df["RiskBand"] = df["ChurnRiskScore"].apply(risk_band)

display(df[["CustomerID","Segment","ChurnRiskScore","RiskBand","Churn"]].sort_values("ChurnRiskScore",ascending=False))


## 5. Validate edge cases and reproducibility

In [ ]:
checks = {
    "Score minimum >= 0": (df["ChurnRiskScore"] >= 0).all(),
    "Score maximum <= 6": (df["ChurnRiskScore"] <= 6).all(),
    "Risk bands valid": df["RiskBand"].isin(["Low","Medium","High"]).all(),
    "Every customer has a segment": df["Segment"].notna().all(),
    "Every customer has an action": True,
    "No duplicate CustomerID": df["CustomerID"].duplicated().sum() == 0
}
display(pd.DataFrame({"Check":checks.keys(),"Passed":checks.values()}))
assert all(checks.values())


## 6. Segment summary

In [ ]:
segment_summary = df.groupby("Segment").agg(
    Customers=("CustomerID","count"),
    Churned=("Churn", lambda s: (s=="Yes").sum()),
    AvgRiskScore=("ChurnRiskScore","mean"),
    AvgMonthlyCharge=("MonthlyCharges","mean"),
    AvgTenure=("TenureMonths","mean"),
    AvgSupportTickets=("SupportTickets","mean")
).reset_index()

segment_summary["ObservedChurnRate"] = (
    segment_summary["Churned"] / segment_summary["Customers"]
)

display(segment_summary.sort_values("ObservedChurnRate",ascending=False))


## 7. Recommendation

**Primary audience:** High-touch monthly customers.

**Decision:** Prioritize proactive support and retention follow-up for this segment.

**Supporting metric:** The segment combines Month-to-Month contracts with 3+ recent support tickets and has the strongest direct prioritization signal in this sample.

**Uncertainty:** Only 15 synthetic customers are available, and the score weights are manually chosen rules rather than statistically learned coefficients.

**Next measurement:** Compare retention/churn and monthly revenue retained after outreach against an appropriate comparison group, then reassess the score using a larger sample.


## 8. Export deliverables

The notebook produces:
- `customer_segments_risk_scores.csv`
- `segment_summary.csv`
- `risk_band_summary.csv`
- `recommendation_memo.md`

These files make the segmentation and score reproducible and reviewable.
